Script for comparing different CNN models 

In [ ]:
import os.path as op
import mne 
import os
from termcolor import colored
import numpy as np 
import pandas as pd
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import classification_report, confusion_matrix,mean_absolute_error,accuracy_score,ConfusionMatrixDisplay, f1_score, classification_report  # Import necessary metrics
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
from matplotlib import pyplot as plt
import tkinter as tk
import glob
import matplotlib
import pickle
import tensorflow as tf

from sklearn.datasets import make_multilabel_classification
from sklearn.preprocessing import MultiLabelBinarizer

from scipy.fft import fft, ifft,fftfreq
from scipy.signal import welch, find_peaks 

from tensorflow.keras.optimizers import Adam
from tensorflow.keras.models import Sequential, Model  # Import Sequential model from TensorFlow Keras
from tensorflow.keras.layers import Conv1D, Conv1DTranspose, MaxPooling1D, Flatten, Dense, Input, Dropout  # Import necessary layers from TensorFlow Keras
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.regularizers import l2, l1

mne.set_log_level("CRITICAL")

In [ ]:
import tensorflow as tf

print(tf.config.list_physical_devices('GPU'))

Loading in Data

In [ ]:
features_all = pd.read_pickle("training_features_18042026.pkl")
features_all = features_all.rename(columns={'Nap Number': 'Nap_ID'}) # making column names match 

trial_info = pd.read_csv('trial_info_duration_2904.csv')

# add duration information 
features_all = features_all.merge(
    trial_info[['Subject', 'Nap_ID', 'Triggers_Order_Nap', 'Duration_1', 'Duration_2']],
    on=['Subject', 'Nap_ID', 'Triggers_Order_Nap'],
    how='left'
)

features_all = features_all.rename(columns={
    'Duration_1': 'Duration_corr',
    'Duration_2': 'Duration_zygo'
})

features_all_store = features_all

x = np.isnan(features_all['Duration_zygo']) 
indices = np.where(x)[0]
features_all = features_all.drop(indices)

In [ ]:
np.shape(features_all_store)

## Functions

In [ ]:
# single head CNN model for number of contractions  
def CNN_model_contraction(input_shape, num_classes,feature_num):
    global epoch_len

    model = Sequential([
        Input(shape=(epoch_len, feature_num)) # NEED TO MAKE SHAPE A PARAMETER 
         # binary
    ])

    # Convolutional Layers - 
    model.add(Conv1D(32, kernel_size=3, activation='relu', input_shape=input_shape))  # Add a 1D convolutional layer with 32 filters and ReLU activation
    model.add(MaxPooling1D(pool_size=2))  # Add a max pooling layer

    model.add(Conv1D(64, kernel_size=3, activation='relu'))  # Add another 1D convolutional layer with 64 filters and ReLU activation
    model.add(MaxPooling1D(pool_size=1))  # Add another max pooling layer

    #model.add(Conv1D(128, kernel_size=3, activation='relu'))  # Add another 1D convolutional layer?

    # Flattening Layer
    model.add(Flatten())  # Flatten the output of the convolutional layers


    # Fully Connected Layers - connect each neuron of one layer to other neuron 
    model.add(Dense(128, activation='relu')) # Add a fully connected layer with 128 neurons and ReLU activation
    model.add(Dropout(0.5)) # see if this changes anything? 
    model.add(Dense(num_classes, activation='softmax'))  # Add the output layer with softmax activation


    return model  # Return the compiled model

In [ ]:
# single head CNN model for duration 
def CNN_model_duration(input_shape, num_classes,feature_num):
    global epoch_len

    model = Sequential([
        Input(shape=(epoch_len, feature_num)) # NEED TO MAKE SHAPE A PARAMETER 
         # binary
    ])

    # Convolutional Layers - 
    model.add(Conv1D(32, kernel_size=3, activation='relu', input_shape=input_shape))  # Add a 1D convolutional layer with 32 filters and ReLU activation
    model.add(MaxPooling1D(pool_size=2))  # Add a max pooling layer

    model.add(Conv1D(64, kernel_size=3, activation='relu'))  # Add another 1D convolutional layer with 64 filters and ReLU activation
    model.add(MaxPooling1D(pool_size=1))  # Add another max pooling layer

    #model.add(Conv1D(128, kernel_size=3, activation='relu'))  # Add another 1D convolutional layer?

    # Flattening Layer
    model.add(Flatten())  # Flatten the output of the convolutional layers


    # Fully Connected Layers - connect each neuron of one layer to other neuron 
    model.add(Dense(128, activation='relu')) # Add a fully connected layer with 128 neurons and ReLU activation
    model.add(Dropout(0.5)) # see if this changes anything? 
    model.add(Dense(2, activation='linear'))


    return model  # Return the compiled model

In [134]:
def CNN_model_twohead(input_shape, num_classes,feature_num):
    global epoch_len

    inputs = Input(shape=(epoch_len, feature_num))

    # Convolutional Layers - 
    x = Conv1D(32, kernel_size=3, activation='relu', input_shape=input_shape)(inputs) # Add a 1D convolutional layer with 32 filters and ReLU activation
    x = MaxPooling1D(pool_size=2)(x) # Add a max pooling layer

    x = Conv1D(64, kernel_size=3, activation='relu')(x)
    x = MaxPooling1D(pool_size=1)(x)

    #model.add(Conv1D(128, kernel_size=3, activation='relu'))  # Add another 1D convolutional layer?

    # Flattening Layer
    x = Flatten()(x)  # Flatten the output of the convolutional layers


 
    shared = Dense(128, activation='relu')(x)
    shared = Dropout(0.5)(shared)

    # Count-specific hidden layers
    count_branch = Dense(32, activation='relu')(shared)
    count_branch = Dropout(0.2)(count_branch)

    count_output = Dense(
        num_classes,
        activation='softmax',
        name='count_output'
    )(count_branch)

    duration_branch = Dense(32, activation='relu')(shared)
    duration_branch = Dropout(0.2)(duration_branch) # try increasing drop out for more regularization? (increase if overfitting)

    duration_output = Dense(
        1,
        activation='linear',
        name='duration_output'
    )(duration_branch)


    model = Model(inputs=inputs, outputs=[count_output, duration_output])

    return model  # Return the compiled model

In [ ]:
def evaluate_model(model, X_train, y_train, X_test, y_test, cvScores, model_name=None):
    # ---- Cross-validation stats ----
    avgScores = np.mean(cvScores)
    stdScores = np.std(cvScores)

    if model_name:
        print(f"\n===== {model_name} =====")

    print(f"Average KFold CV Score: {avgScores:.4f}")
    print(f"Std KFold CV Score: {stdScores:.4f}")

    # ---- Predictions ----
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)

    # Handle two-head model (take first output = count head)
    if isinstance(y_pred_train, list):
        y_pred_train = y_pred_train[0]
        y_pred_test = y_pred_test[0]

    # Convert probabilities → class labels
    y_pred_train = np.argmax(y_pred_train, axis=1)
    y_pred_test = np.argmax(y_pred_test, axis=1)

    # ---- Metrics ----
    accuracy_training = accuracy_score(y_train, y_pred_train)
    accuracy_test = accuracy_score(y_test, y_pred_test)

    f1_training = f1_score(y_train, y_pred_train, average='weighted')
    f1_test = f1_score(y_test, y_pred_test, average='weighted')

    # ---- Print results ----
    print(f"Training Accuracy: {accuracy_training:.4f}")
    print(f"Test Accuracy: {accuracy_test:.4f}")
    print(f"Training F1 Score: {f1_training:.4f}")
    print(f"Test F1 Score: {f1_test:.4f}")

    # ---- Return results (useful for logging/comparison) ----
    return {
        "cv_mean": avgScores,
        "cv_std": stdScores,
        "train_acc": accuracy_training,
        "test_acc": accuracy_test,
        "train_f1": f1_training,
        "test_f1": f1_test
    }

## Single Channel Training 

Single Head Count 

In [ ]:
# Define CNN model inputs 
subj_omit = False # omiting certain subjects on training - replace with string 

if (subj_omit):
    features_all_temp = features_all.loc[features_all["Subject"] != subj_omit]

else:
    features_all_temp = features_all


X_zygo = features_all_temp[["Zygo"]] 
X_corr = features_all_temp[["Corr"]]

X_zygo = X_zygo.to_numpy()
X_zygo = np.array(X_zygo.tolist())
X_zygo = np.transpose(X_zygo, (0, 2, 1)) # setting dimensions 

X_corr = X_corr.to_numpy()
X_corr = np.array(X_corr.tolist())
X_corr = np.transpose(X_corr, (0, 2, 1)) # setting dimensions 

X = np.concatenate((X_zygo, X_corr), axis=0)
y = np.concatenate([features_all_temp["Num_Contractions_Zygo"].astype(int).to_numpy(),
                    features_all_temp["Num_Contractions_Corr"].astype(int).to_numpy()])       


indices = np.arange(len(X))
idx_train, idx_test = train_test_split(indices, test_size=0.2, random_state=42) # keep 20% purely for testing 

# split like this to be able to recover the indices for later when rescoring 
X_train_full = X[idx_train]
X_test = X[idx_test]
y_train_full = y[idx_train]
y_test = y[idx_test]

input_shape = X_train_full.shape[1:]   
num_classes = len(np.unique(y))    

input_shape = (input_shape[0], 1)   
feature_num = np.shape(X_zygo)[2]
epoch_len = np.shape(X_zygo)[1]

In [ ]:
kf = KFold(n_splits=5, random_state = 42, shuffle=True) # 5 folds 
cvScores_contraction=[]
epoch_num = 10 
k = 1

for train_index, test_index in kf.split(X_train_full):
    print(f"Fold: {k} ==================================================================")
    
    X_train, X_val = X[train_index], X[test_index]
    y_train, y_val = y[train_index], y[test_index]

    model_contraction = CNN_model_contraction(input_shape, num_classes,feature_num)
    model_contraction.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])  

    #early_stop = EarlyStopping(monitor='val_loss',  patience=3, restore_best_weights=True)
    
    model_history_kfold = model_contraction.fit(X_train, y_train, epochs=epoch_num, validation_data=(X_val, y_val)) #, callbacks=[early_stop])
    #plot_accuracy(model_history_kfold,i)
    
    scores = model_contraction.evaluate(X_test,y_test)
    cvScores_contraction.append(scores[1] * 100)

    k += 1 
    

model_history_contraction = model_contraction.fit(X_train_full, y_train_full, epochs=epoch_num, validation_data=(X_test, y_test)) #, callbacks=[early_stop])

In [ ]:
# cross validation results 
avgScores = np.mean(cvScores_contraction)
stdScores = np.std(cvScores_contraction)

print(f"Average KFold Cross Validation Score: {avgScores}")
print(f"Standard Deviation KFold Cross Validation Score: {stdScores}")

# full training results (test data not seen during cross val)
y_pred_train = model_contraction.predict(X_train_full)  
y_pred_train = np.argmax(y_pred_train, axis=1)   

# Predict on test data
y_pred_test = model_contraction.predict(X_test)   
y_pred_test = np.argmax(y_pred_test, axis=1)   

# Calculate accuracy
accuracy_training = accuracy_score(y_train_full, y_pred_train)   
accuracy_test = accuracy_score(y_test, y_pred_test)  

# Calculate F1 score
f1_training = f1_score(y_train_full, y_pred_train, average='weighted')  
f1_test = f1_score(y_test, y_pred_test, average='weighted')  

# Print accuracy and F1 score
print("Training Accuracy :", accuracy_training)  
print("Test Accuracy :", accuracy_test)  
print("Training F1 Score :", f1_training)   
print("Test F1 Score :", f1_test) 

Single Head Duration

In [ ]:
# Define CNN model inputs
subj_omit = False # omiting certain subjects on training - replace with string 

if (subj_omit):
    features_all_temp = features_all.loc[features_all["Subject"] != subj_omit]

else:
    features_all_temp = features_all


X_zygo = features_all_temp[["Zygo"]] 
X_corr = features_all_temp[["Corr"]]

X_zygo = X_zygo.to_numpy()
X_zygo = np.array(X_zygo.tolist())
X_zygo = np.transpose(X_zygo, (0, 2, 1)) # setting dimensions 

X_corr = X_corr.to_numpy()
X_corr = np.array(X_corr.tolist())
X_corr = np.transpose(X_corr, (0, 2, 1)) # setting dimensions 

X = np.concatenate((X_zygo, X_corr), axis=0)
y = np.concatenate([features_all_temp["Duration_zygo"],
                    features_all_temp["Duration_corr"]])       


indices = np.arange(len(X))
idx_train, idx_test = train_test_split(indices, test_size=0.2, random_state=42) # keep 20% purely for testing 

# split like this to be able to recover the indices for later when rescoring 
X_train_full = X[idx_train]
X_test = X[idx_test]
y_train_full = y[idx_train]
y_test = y[idx_test]

input_shape = X_train_full.shape[1:]   
num_classes = len(np.unique(y))    

input_shape = (input_shape[0], 1)   
feature_num = np.shape(X_zygo)[2]
epoch_len = np.shape(X_zygo)[1]

In [ ]:
kf = KFold(n_splits=5, random_state = 42, shuffle=True) # 5 folds 
cvScores_duration=[]
epoch_num = 10 
k = 1
 
for train_index, test_index in kf.split(X_train_full):
    print(f"Fold: {k} ==================================================================")
    
    X_train, X_val = X[train_index], X[test_index]
    y_train, y_val = y[train_index], y[test_index]


    model_duration = CNN_model_duration(input_shape, num_classes,feature_num)
    model_duration.compile(optimizer='adam', loss='mae', metrics=['mae'])  

    #early_stop = EarlyStopping(monitor='val_loss',  patience=3, restore_best_weights=True)
    
    model_history_kfold = model_duration.fit(X_train, y_train, epochs=epoch_num, validation_data=(X_val, y_val)) #, callbacks=[early_stop])
    #plot_accuracy(model_history_kfold,i)
    
    scores = model_duration.evaluate(X_test,y_test)
    cvScores_duration.append(scores[1])

    k += 1 
    

model_history_duration = model_duration.fit(X_train_full, y_train_full, epochs=epoch_num, validation_data=(X_test, y_test)) #, callbacks=[early_stop])

In [ ]:
np.shape(y_train_full) 
np.shape(y) 

In [ ]:
# cross validation results 
cvScores_duration = np.array(cvScores_duration)
cvScores_duration /= 100 

avgScores = np.mean(cvScores_duration)
stdScores = np.std(cvScores_duration)

print(f"Average KFold Cross Validation Score: {avgScores}")
print(f"Standard Deviation KFold Cross Validation Score: {stdScores}")
 
baseline = np.mean(y_train_full)
mae_baseline = np.mean(np.abs(y_train_full - baseline))

# full training results (test data not seen during cross val)
y_pred_train_dur = model_duration.predict(X_train_full) 

# Predict on test data
y_pred_test_dur = model_duration.predict(X_test) 

  
mae_training_dur = mean_absolute_error(y_train_full, y_pred_train)   
mae_test_dur= mean_absolute_error(y_test, y_pred_test)  
  
# Print accuracy and F1 score
print("Baseline MAE:", mae_baseline)
print("Training MAE :", mae_training_dur)
print("Test MAE :", mae_test_dur)
 

Two Head 

In [ ]:
# Define CNN model inputs 
subj_omit = False # omiting certain subjects on training - replace with string 

if (subj_omit):
    features_all_temp = features_all.loc[features_all["Subject"] != subj_omit]

else:
    features_all_temp = features_all


X_zygo = features_all_temp[["Zygo"]] 
X_corr = features_all_temp[["Corr"]]

X_zygo = X_zygo.to_numpy()
X_zygo = np.array(X_zygo.tolist())
X_zygo = np.transpose(X_zygo, (0, 2, 1)) # setting dimensions 

X_corr = X_corr.to_numpy()
X_corr = np.array(X_corr.tolist())
X_corr = np.transpose(X_corr, (0, 2, 1)) # setting dimensions 

X = np.concatenate((X_zygo, X_corr), axis=0)
y_contractions = np.concatenate([features_all_temp["Num_Contractions_Zygo"].astype(int).to_numpy(),
                    features_all_temp["Num_Contractions_Corr"].astype(int).to_numpy()])       

y_durations = np.concatenate([features_all_temp["Duration_zygo"],
                    features_all_temp["Duration_corr"]])       

y = np.column_stack((y_contractions, y_durations))

indices = np.arange(len(X))
idx_train, idx_test = train_test_split(indices, test_size=0.2, random_state=42) # keep 20% purely for testing 

# split like this to be able to recover the indices for later when rescoring 
X_train_full = X[idx_train]
X_test = X[idx_test]
y_train_full = y[idx_train]
y_test = y[idx_test]

input_shape = X_train_full.shape[1:]   
num_classes = len(np.unique(y))    

input_shape = (input_shape[0], 1)   
feature_num = np.shape(X_zygo)[2]
epoch_len = np.shape(X_zygo)[1]

In [135]:
# Create CNN model using the adjusted input shape and number of classes
kf = KFold(n_splits=5, random_state = 42, shuffle=True) # 5 folds 

cvScores_dur =[]
cvScores_contr =[]
cvScores = [] 

epoch_num = 10 
k = 1

for train_index, test_index in kf.split(X_train_full):
    print(f"Fold: {k} ==================================================================")
    
    X_train, X_val = X[train_index], X[test_index]
    y_train, y_val = y[train_index], y[test_index]

    model_twohead = CNN_model_twohead(input_shape, num_classes,feature_num) #CNN_model_regression if treating both as continuous
    model_twohead.compile(optimizer='adam',loss=['sparse_categorical_crossentropy', 'mae'],metrics=['accuracy', 'mae'] ) #think about metric 
    model_history_kfold = model_twohead.fit(X_train,[y_train[:,0], y_train[:,1]], 
                                            validation_data=(X_val,[y_val[:,0], y_val[:,1]]), 
                                            epochs=epoch_num)
    
    scores = model_twohead.evaluate(X_test,[y_test[:,0], y_test[:,1]])
    metrics = dict(zip(model_twohead.metrics_names, scores))

    cvScores_dur.append(metrics['duration_output_mae'])
    cvScores_contr.append(metrics['count_output_accuracy'] * 100)


    cvScores.append(scores)
 
    

  
    #early_stop = EarlyStopping(monitor='val_loss',  patience=3, restore_best_weights=True)
    #model_history_kfold = model.fit(X_train, y_train, epochs=epoch_num, validation_data=(X_val, y_val)) #, callbacks=[early_stop])
    
    #plot_accuracy(model_history_kfold,i)
    
    

    k += 1 
    

model_history_twohead = model_twohead.fit(X_train_full, [y_train_full[:,0], y_train_full[:,1]], epochs=epoch_num, validation_data=(X_test, [y_test[:,0], y_test[:,1]])) #, callbacks=[early_stop])

Fold: 1 ==================================================================
Epoch 1/10
286/286 [==============================] - 21s 59ms/step - loss: 99.8607 - count_output_loss: 8.4307 - duration_output_loss: 91.4301 - count_output_accuracy: 0.5274 - count_output_mae: 0.7995 - duration_output_accuracy: 0.3469 - duration_output_mae: 91.4301 - val_loss: 8.9409 - val_count_output_loss: 1.6111 - val_duration_output_loss: 7.3298 - val_count_output_accuracy: 0.7339 - val_count_output_mae: 0.7731 - val_duration_output_accuracy: 0.0039 - val_duration_output_mae: 7.3298
Epoch 2/10
286/286 [==============================] - 17s 58ms/step - loss: 26.1231 - count_output_loss: 1.7113 - duration_output_loss: 24.4118 - count_output_accuracy: 0.7271 - count_output_mae: 0.7995 - duration_output_accuracy: 0.3257 - duration_output_mae: 24.4118 - val_loss: 5.5023 - val_count_output_loss: 1.2447 - val_duration_output_loss: 4.2576 - val_count_output_accuracy: 0.7628 - val_count_output_mae: 0.7731 - val_du

In [ ]:
cvScores

In [136]:
# comparison for two head model 
# cross validation results 
avgScores_dur = np.mean(cvScores,axis=0)[3]
stdScores_dur = np.std(cvScores,axis=0)[3]

avgScores_contr= np.mean(cvScores,axis=0)[6]
stdScores_contr = np.std(cvScores,axis=0)[6]
 

print(f"Average KFold Cross Validation Score for contraction: {avgScores_contr}")
print(f"Standard Deviation KFold Cross Validation Score for contractionon: {stdScores_contr}")

print(f"Average KFold Cross Validation Score for duration: {avgScores_dur}")
print(f"Standard Deviation KFold Cross Validation Score for duration: {avgScores_dur}")

# full training results (test data not seen during cross val)
[y_pred_train_contraction, y_pred_train_dur] = model_twohead.predict(X_train_full)  

y_pred_train_contraction = np.argmax(y_pred_train_contraction, axis=1)   

# Predict on test data
[y_pred_test_contraction, y_pred_test_dur] = model_twohead.predict(X_test)  

y_pred_test_contraction = np.argmax(y_pred_test_contraction, axis=1)   

# Calculate accuracy
accuracy_training_contraction = accuracy_score(y_train_full[:,0], y_pred_train_contraction)   
accuracy_test_contraction = accuracy_score(y_test[:,0], y_pred_test_contraction)  

# Calculate F1 score
f1_training_contraction = f1_score(y_train_full[:,0], y_pred_train_contraction, average='weighted')  
f1_test_contraction = f1_score(y_test[:,0], y_pred_test_contraction, average='weighted')  

# MAE 
baseline = np.mean(y_train_full[:,1])
mae_baseline = np.mean(np.abs(y_train_full[:,1] - baseline))

y_pred_train_dur = model_twohead.predict(X_train_full) 
y_pred_test_dur = model_twohead.predict(X_test) 

  
mae_training_dur = mean_absolute_error(y_train_full[:,1], y_pred_train)   
mae_test_dur= mean_absolute_error(y_test[:,1], y_pred_test)  
  
# Print accuracy and F1 score

# Print accuracy and F1 score
print("Training Accuracy :", accuracy_training_contraction)  
print("Test Accuracy :", accuracy_test_contraction)  
print("Training F1 Score :", f1_training_contraction)   
print("Test F1 Score :", f1_test_contraction)   
print("----------------------") 
print("Baseline MAE:", mae_baseline)
print("Training MAE :", mae_training_dur)
print("Test MAE :", mae_test_dur)


Average KFold Cross Validation Score for contraction: 3.3676106929779053
Standard Deviation KFold Cross Validation Score for contractionon: 0.4367369162855505
Average KFold Cross Validation Score for duration: 0.8709383726119995
Standard Deviation KFold Cross Validation Score for duration: 0.8709383726119995
90/90 [==============================] - 1s 9ms/step
Training Accuracy : 0.8929446778711485
Test Accuracy : 0.8778011204481793
Training F1 Score : 0.8702639273170646
Test F1 Score : 0.8563687215746459
----------------------
Baseline MAE: 5.9552841687640035
Training MAE : 3.4120064611866225
Test MAE : 3.429312634327922


## Two Channel 

Single Head Duration 

Single Head Count 

Two Head 

## Segmentation

In [ ]:
def comparator(learner, instructor):
    if len(learner) != len(instructor):
        raise AssertionError("Layer count mismatch")
    for a, b in zip(learner, instructor):
        if tuple(a) != tuple(b):
            print(colored("Test failed", attrs=['bold']))
            raise AssertionError("Error in test")
    print(colored("All tests passed!", "green"))

def summary(model):
    result = []
    for layer in model.layers:
        output_shape = getattr(layer.output, 'shape', None)
        params = layer.count_params() if hasattr(layer, 'count_params') else 0
        result.append([layer.__class__.__name__, output_shape, params])
    return result
